# P7 - Pipeline End-to-End (main.py)

Integración completa de todos los módulos (P1-P6) mediante una máquina de estados autónoma.

# ─────────────────────────────────────────────────────
# SETUP INICIAL
# ─────────────────────────────────────────────────────
import sys
import os
import time
import numpy as np

# Agregar ruta a P7_PIPELINE
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'P7_PIPELINE'))

print('Rutas configuradas ✓')

# ─────────────────────────────────────────────────────
# CONEXION AL ROBOT
# ─────────────────────────────────────────────────────
from pymycobot.mycobot import MyCobot

mc = MyCobot('/dev/ttyUSB0', 1000000)
mc.power_on()
time.sleep(1)
assert mc.is_controller_connected(), 'Error de conexión'
print('Robot conectado ✓')

# ─────────────────────────────────────────────────────
# IMPORTAR MAIN.PY CON TODA LA INTEGRACION P1-P6
# ─────────────────────────────────────────────────────
from main import State, CycleResult, PipelineController

print('Pipeline importado ✓')

## Verificar FSM y estados

In [ ]:
print("="*70)
print("ESTADOS DE LA FSM")
print("="*70)

for state in State:
    print(f"  • {state.value}")

print("\n✓ FSM lista para ejecución")

## Verificar módulos integrados

In [ ]:
from main import (
    P1_AVAILABLE, P2_AVAILABLE, P3_AVAILABLE,
    P4_AVAILABLE, P5_AVAILABLE, P6_AVAILABLE
)

print("="*70)
print("MÓDULOS DISPONIBLES")
print("="*70)

modules = [
    ('P1 (DH)', P1_AVAILABLE),
    ('P2 (FK)', P2_AVAILABLE),
    ('P3 (IK)', P3_AVAILABLE),
    ('P4 (Colisiones)', P4_AVAILABLE),
    ('P5 (Control)', P5_AVAILABLE),
    ('P6 (Visión)', P6_AVAILABLE)
]

for name, available in modules:
    status = "✓" if available else "✗"
    print(f"  [{status}] {name}")

## Ejecutar 5 ciclos autónomos

In [ ]:
print("="*70)
print("EJECUCIÓN: 5 CICLOS AUTÓNOMOS DE PICK-PLACE")
print("="*70)

# Inicializar controlador
controller = PipelineController(robot=mc)

# Ejecutar 5 ciclos
results = []
for cycle_num in range(1, 6):
    print(f"\n[CICLO {cycle_num}]")
    try:
        result = controller.run_cycle()
        results.append(result)
        print(f"  ✓ Ciclo completado")
        print(f"    - Exitoso: {result.success}")
        print(f"    - Errores: {result.errors}")
        print(f"    - Duración: {result.duration_sec:.2f}s")
    except Exception as e:
        print(f"  ✗ Error: {e}")
    
    if cycle_num < 5:
        time.sleep(2)

print("\n" + "="*70)

## Análisis de resultados

In [ ]:
print("="*70)
print("RESUMEN DE RESULTADOS")
print("="*70)

if results:
    successful = sum(1 for r in results if r.success)
    total = len(results)
    success_rate = (successful / total) * 100
    total_errors = sum(r.errors for r in results)
    avg_duration = np.mean([r.duration_sec for r in results])
    
    print(f"\nCiclos exitosos: {successful}/{total}")
    print(f"Tasa de éxito: {success_rate:.1f}%")
    print(f"Errores totales: {total_errors}")
    print(f"Duración promedio: {avg_duration:.2f}s")
    
    print(f"\nDetalles por ciclo:")
    for i, result in enumerate(results, 1):
        print(f"  Ciclo {i}: {'✓' if result.success else '✗'} ({result.duration_sec:.2f}s, {result.errors} errores)")
else:
    print("⚠ No se ejecutaron ciclos")

print("\n✓ Verificar sesion.log para detalles")